# RAG

In [1]:
!pip -q install langchain_community rank_bm25 langchain-huggingface langchain sentence_transformers PyPDF2 pycryptodome tiktoken faiss-gpu


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 255.2/255.2 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.5/85.5 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.4/404.4 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.8/295.8 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.0/78.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3

In [19]:
# create a faiss_vector_store for testing
pdf_reader = PdfReader('/content/FINAL-Audit-Report-2023.pdf')
text = ""
chunks=[]

for page in pdf_reader.pages:
    text += page.extract_text()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks=text_splitter.create_documents([text])
embedding_model = HuggingFaceEmbeddings(model_name='sentence-transformers/all-mpnet-base-v2')
faiss_vector_store = FAISS.from_documents(chunks, embedding_model)

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [50]:
import os
import re
from PyPDF2 import PdfReader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.retrievers import BM25Retriever, EnsembleRetriever

class RAG:
    def __init__(self, pdf_path, method, faiss_vector_store=None):

        self.text = self.read_pdf(pdf_path)
        self.chunks=self.get_chunks(pdf_path, method)
        self.embedding_model = HuggingFaceEmbeddings(model_name='sentence-transformers/all-mpnet-base-v2')

        # initialize different retriever
        if faiss_vector_store is None: # faiss store is slow to create if the embedding model is large, so we load a pre-computed one
            self.faiss_vector_store = FAISS.from_documents(self.chunks, self.embedding_model)
        else:
            self.faiss_vector_store = faiss_vector_store
        self.faiss_retriever = self.faiss_vector_store.as_retriever(search_kwargs={"k": 3})

        self.bm25_retriever = BM25Retriever.from_documents(self.chunks)
        self.bm25_retriever.k =  3

        self.ensemble_retriever = EnsembleRetriever(retrievers=[self.faiss_retriever,self.bm25_retriever], weights=[0.5, 0.5])

    def get_chunks(self, pdf_path, method):
        if method=='unstructured': # the package unstructured
            return self.get_chunks_unstructured(pdf_path)

        elif method=='recursive':
            text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
            chunks=text_splitter.create_documents([self.text])
            return chunks

    def get_chunks_unstructured(self, pdf_path):
        return None

    def read_pdf(self, pdf_path):
        pdf_reader = PdfReader(pdf_path)
        text = ""
        for page in pdf_reader.pages:
            text += page.extract_text()
        return text

    def search(self, query, method):
        if method=='ensemble':
            return self.search_return_ensemble(query)
        elif method=='all':
            return self.search_return_all()

    def search_return_ensemble(self, query):
        documents=self.get_documents_ensemble(query)
        text=""
        for d in documents[:3]:
            text+=d.page_content
        return text

    def search_return_all(self):
        return self.text

    def get_documents_ensemble(self, query):
        return self.ensemble_retriever.invoke(input=query)



rag=RAG(pdf_path='/content/FINAL-Audit-Report-2023.pdf',method='recursive', faiss_vector_store=faiss_vector_store)



/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [41]:
text=rag.search(query='total revenue', method='ensemble')
print(text)

Total Revenue 979,651           -                        979,651           
Net Assets Released from Restrictions 975,954           (975,954)          -                        
Total Support and Revenue 3,546,026       638,761           4,184,787       
Expenses
Program services
Research 1,574,354       -                        1,574,354       
Education and service 1,146,980       -                        1,146,980       
Drug Development Collaborative 1,188,427       -                        1,188,427       
Total Program Services 3,909,761       -                        3,909,761       
Supporting services
Management and general 538,079           -                        538,079           
Fundraising 548,618           -                        548,618           
Total Supporting Services 1,086,697       -                        1,086,697       
Total Expenses 4,996,458       -                        4,996,458agreements are signed annually. Earned income received in advance of the co

# React

In [27]:
!pip -q install openai python-dotenv


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.2/325.2 kB 21.2 MB/s eta 0:00:00


In [28]:
%%writefile .env
OPENAI_API_KEY=""

Writing .env


In [30]:
from dotenv import load_dotenv
load_dotenv()

True

In [31]:
import os
from openai import OpenAI
import re
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))


messages = [
        {"role": "system", "content": 'You are a sport fanatic'},
        {"role": "user", "content": 'how many superbowls has Rams won'},
    ]
response = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        temperature=0
    )
response_message = response.choices[0].message.content
print(response_message )



As of October 2023, the Los Angeles Rams have won two Super Bowls. Their first victory came in Super Bowl XXXIV in 2000 when they were based in St. Louis, and their second win was in Super Bowl LVI in 2022 after they had returned to Los Angeles.


In [32]:
class Agent:
    def __init__(self, client=client, system_prompt: str = "") -> None:
        self.client = client # the llm
        self.system_prompt = system_prompt
        self.messages: list = []
        if self.system_prompt:
            self.messages.append({"role": "system", "content": self.system_prompt})

    def __call__(self, message=""):
        if message:
            self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        completion = client.chat.completions.create(
            model="gpt-4o",
            messages=self.messages,
            temperature=0
        )
        return completion.choices[0].message.content

In [33]:
system_prompt = """
you are a financial expert and an underwriter working for commerical banking of a bank.
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

search:
e.g. search: total income
Runs a serach function in the knowledge base and return relevant information as chunks of text.


Example session:

Question: analyze financial performance of Apple Inc and potential drivers?
Thought: to analyse finanial performance, I need to know Apple's total revenue, so I need to search its total revenue
Action: search: Apple Inc total revenue
PAUSE

You will be called again with this:

Observation: 'Total Revenue 123,456...'

Thought: to know the potential drivers, I need to know the buisness composition of Apple, so I need to search its buisness composition
Action: search: Apple Inc buisness composition
PAUSE

You will be called again with this:

Observation: 'Apple's business composition includes iPhone: Major revenue driver. Mac: Laptops and desktops...'

If you have the answer, output it as the Answer.

Answer: Apple's financial performance is strong with a total revenue of 123,456, which is drived by iPhone sales.

Now it's your turn:
"""

In [34]:
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))


In [46]:
len(rag.text)

49789

In [53]:
class ReactExcecutor:
    def __init__(self, client, tools, system_prompt, max_iterations=10):

        self.client = client
        self.tools = tools
        self.system_prompt = system_prompt
        self.max_iterations = max_iterations

    def run(self, query):
        agent = Agent(client = self.client, system_prompt=self.system_prompt)
        next_prompt = query
        i = 0
        while i < self.max_iterations:
            i += 1
            result = agent(next_prompt)


            print(result)


            if "PAUSE" in result and "Action" in result:
                action = re.findall(r"Action: ([a-z_]+): (.+)", result, re.IGNORECASE)
                chosen_tool = action[0][0]
                arg = action[0][1]

                if chosen_tool in self.tools:
                    result_tool = eval(f"{chosen_tool}('{arg}')")
                    next_prompt = f"Observation: {result_tool}"

                else:
                    next_prompt = "Observation: Tool not found"
                print("-"*80)
                if not ("Observation:" in next_prompt and len(next_prompt)>40000):
                    print(next_prompt)
                print()
                continue

            if "Answer" in result:
                break


# use ensemble top 3 results to search

In [43]:
def search(query):
    return rag.search(query, method='ensemble')
reactExcecutor = ReactExcecutor(client=client, tools=["search"], system_prompt=system_prompt)


In [38]:
reactExcecutor.run(query="what is the total revenue of National Ataxia Foundation in 2023")

Thought: To provide the total revenue of the National Ataxia Foundation in 2023, I need to search for their financial statements or reports for that year.
Action: search: National Ataxia Foundation total revenue 2023
PAUSE
--------------------------------------------------------------------------------
Observation: With donor restriction 2,680,567       2,041,806       
Total Net Assets 3,614,946       4,426,617       
Total Liabilities and Net Assets 4,116,826$     5,008,595$      National Ataxia Foundation
Statements of Financial Position
December 31, 2023 and 2022
See Independent Auditor's Report and Notes to the Financial Statements.
6Without Donor With Donor
Restriction Restriction Total
Support and Revenue
Support
Contributions, memorials and honorariums 1,468,504$     1,614,715$      3,083,219$      
In-kind contributions 121,917           -                        121,917           
Total Support 1,590,421       1,614,715       3,205,136       
Revenue
Conference income 264,816 

In [39]:
query ="How did deferred revenue change between 2022 and 2023, and what drove this change?"
reactExcecutor.run(query=query)

Thought: To analyze the change in deferred revenue between 2022 and 2023, I need to find the deferred revenue figures for both years. I will start by searching for the deferred revenue in 2022.
Action: search: deferred revenue 2022
PAUSE
--------------------------------------------------------------------------------
Observation: agreements are signed annually. Earned income received in advance of the collaborative period is reported as deferred 
revenue. In the case of conference income, revenue is recognized at the time the event is held, which is at a point in time. 
Conference income received in advance of the event is reported as deferred revenue.  
 
• Performance Obligations  
 
The performance obligation related to Drug D evelopment Collaborative is satisfied over the collaborative period; 
therefore, the Foundation  recognizes revenue over a period of time on a monthly basis. The performance 
obligation related to the conference income is satisfied once the event is held; ther

the number returned is not correct, what is being returned is 'deferred conference revenue', not 'deferred revenue'

In [44]:
query="What caused the significant change in net assets with donor restrictions between 2022 and 2023?"
reactExcecutor.run(query=query)

Thought: To determine the cause of the significant change in net assets with donor restrictions between 2022 and 2023, I need to find information on the financial statements or reports that detail these changes. This may include donations received, changes in donor restrictions, or expenditures related to these assets. I will start by searching for the financial statements or reports for the years 2022 and 2023.
Action: search: net assets with donor restrictions 2022 2023 financial statements
PAUSE
--------------------------------------------------------------------------------
Observation: accounting principles generally accepted in the United States of America.   
 
Revenues are recorded when earned and expenses are recorded when a liability is incurred. Contributions received are 
recorded as an increase in non -donor -restricted or donor -restricted support depending on the existence or nature of any 
donor restrictions.  Accordingly, net assets of the Foundation and changes therei

llm does't go the auditor note

In [45]:

query="What was the operating lease liability at the end of 2023, and how does the lease structure affect NAF’s financial obligations?"
reactExcecutor.run(query=query)

Thought: To answer this question, I need to find the operating lease liability for NAF at the end of 2023. I will start by searching for this information.
Action: search: NAF operating lease liability end of 2023
PAUSE
--------------------------------------------------------------------------------
Observation: for fixed monthly base payments starting at $2,563 per month and incrementally increasing to $2,864 per month until 
expiration in September  2025. The agreement also provides that the Foundation  is responsible for a proportional share of 
the property's incurred ownership taxes and operating expenses. The Foundation  has determined that this lease is an 
operating lease.  
 
The Foundation  adopted FASB ASC 842, effective January 1, 2022, using a modified retrospective approach. As a result, 
the Foundation  was required to recognize a ROU asset and corresponding lease liability on the face of the statement of 
financial position for the year ended December 31, 2023  and 2022 

this is correct

# pass entire pdf text to it

In [54]:
def search(query):
    return rag.search(query, method='all')
reactExcecutor = ReactExcecutor(client=client, tools=["search"], system_prompt=system_prompt)


In [55]:
query ="How did deferred revenue change between 2022 and 2023, and what drove this change?"
reactExcecutor.run(query=query)

Thought: To answer this question, I need to find the deferred revenue figures for 2022 and 2023, and then identify the factors that contributed to any changes. I will start by searching for the deferred revenue for 2022.
Action: search: deferred revenue 2022
PAUSE
--------------------------------------------------------------------------------

Thought: I have found the deferred revenue figures for both 2022 and 2023. In 2022, the deferred revenue was $22,291, and in 2023, it increased to $109,810. Now, I need to identify the factors that drove this change in deferred revenue. I will search for any relevant information regarding the drivers of deferred revenue changes.

Action: search: drivers of deferred revenue change 2022 to 2023
PAUSE
--------------------------------------------------------------------------------

Thought: The increase in deferred revenue from $22,291 in 2022 to $109,810 in 2023 was primarily driven by the introduction of deferred revenue from the Drug Development

correct

In [56]:
query="What caused the significant change in net assets with donor restrictions between 2022 and 2023?"
reactExcecutor.run(query=query)

Thought: To determine the cause of the significant change in net assets with donor restrictions between 2022 and 2023, I need to find information on the financial statements or reports that detail these changes. This may include donations received, changes in donor restrictions, or expenditures related to restricted assets. I will start by searching for the financial statements or reports for the years 2022 and 2023.

Action: search: net assets with donor restrictions 2022 2023 financial statements
PAUSE
--------------------------------------------------------------------------------

Thought: The significant change in net assets with donor restrictions between 2022 and 2023 for the National Ataxia Foundation can be attributed to several factors. I need to analyze the specific changes in the components of net assets with donor restrictions as detailed in the financial statements and notes.

Action: search: changes in net assets with donor restrictions National Ataxia Foundation 2022 20

correct

In [57]:
query="What was the operating lease liability at the end of 2023, and how does the lease structure affect NAF’s financial obligations?"
reactExcecutor.run(query=query)

Thought: To answer this question, I need to find the operating lease liability for NAF at the end of 2023. I will start by searching for this information.
Action: search: NAF operating lease liability end of 2023
PAUSE
--------------------------------------------------------------------------------

Thought: I have found the operating lease liability information for NAF at the end of 2023. The current portion of the operating lease liability is $32,750, and the long-term portion is $25,353. Now, I need to analyze how the lease structure affects NAF's financial obligations.

Action: search: impact of operating lease structure on financial obligations
PAUSE
--------------------------------------------------------------------------------

Thought: The lease structure affects NAF's financial obligations by creating a fixed liability that must be paid over the lease term. The operating lease liability is recognized on the balance sheet, which impacts the organization's financial position by

correct

In [58]:
print(1)

1
